# Step 4 — Richer causal features (personalized PageRank from illicit + temporal motifs)

Step 3's GNN underperformed the hand-engineered baseline. Pivoting to two new feature families that are maximally orthogonal to the existing 103 trajectory + 17 Block B pair features.

## Block C — Personalized PageRank seeded from illicit history

**Question being asked**: "How reachable is T from the illicit subgraph that exists at time `t(T)`?"

For each timestep `t`, run personalized PageRank on the **cumulative causal bipartite graph** at `t` (all addr↔tx edges with `τ ≤ t`), with the teleport vector concentrated on **illicit txs at `τ < t(T)`**. The resulting score for tx `T` is a single scalar that captures multi-hop influence from the illicit subgraph that the per-wallet trajectory features can't express directly.

Per-tx features extracted:
- `ppr_illicit`: T's PPR mass from illicit seed.
- `ppr_illicit_log`: log(1 + ppr_illicit) for scale stability.
- `max_nbr_ppr_illicit`: max PPR mass over T's incident wallets.
- `mean_nbr_ppr_illicit`: mean PPR mass over T's incident wallets.
- `n_illicit_seeds_at_t`: cardinality of seed set at `t` (a proxy for how informative the score is).

## Block D — Temporal motifs (2-hop temporal paths terminating at T)

For each tx `T` at time `t`, count cross-step 2-hop temporal paths `T'' → T' → T` where edges are (T'' → T' → T) chained along the cross-step tx→tx graph. Two timesteps must be strictly increasing (`t(T'') < t(T') < t(T)`).

Counts broken out by:
- total 2-hop count
- count where T'' is illicit
- count where T' is illicit
- count where both edges have role `out→in` (clean money-flow chain)
- count where the 2-hop chain has total Δt ≤ 5 (recent peeling-like pattern)
- decayed sum: `Σ exp(-β · (t − t(T''))) · 1[T'' illicit]`

These are explicit 2-hop signals; the 1-hop pair features in Block B can't see T''.

## Ablation grid
| | Features | dim |
|---|---|---|
| A0 | 108 intrinsic | 108 |
| A1 | + 103 traj | 211 |
| A3 | + Block B (17) | 228 |
| **C1** | A3 + Block C (5 PPR) | 233 |
| **C2** | A3 + Block D (6 motifs) | 234 |
| **C3** | A3 + Block C + Block D | 239 |

Same temporal split, same RF hyperparameters.

In [1]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, classification_report

ROOT = Path.cwd()
while not (ROOT / "transactions_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR      = ROOT / "actors_data"
print(f"repo root: {ROOT}")

TRAIN_END    = 34
N_TIME_STEPS = 49
RANDOM_SEED  = 175
MAX_WALLET_DEGREE = 50
PPR_DAMP        = 0.85
PPR_ITERATIONS  = 30
MOTIF_DECAY     = 0.2
np.random.seed(RANDOM_SEED)

repo root: /Users/bryanfang/Library/CloudStorage/OneDrive-HarvardUniversity/School/2025-2026/STAT 175/175_final_project


In [2]:
print("Loading data...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.txt")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.txt")
tx_classes_df["label"] = tx_classes_df["class"].map({1:1,2:0,3:-1}).astype(np.int8)
tx_id_to_idx = {tid:i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
all_cols = list(tx_features_df.columns)
GRAPH_COLS = [c for c in all_cols if c.startswith("Aggregate_feature")] + ["in_txs_degree","out_txs_degree"]
feat_cols_intr = [c for c in all_cols if c not in ("txId","Time step") and c not in GRAPH_COLS]
feat_cols_full = [c for c in all_cols if c not in ("txId","Time step")]
F_INTRINSIC = len(feat_cols_intr)
F_FULL = len(feat_cols_full)
merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_intr = tx_features_df[feat_cols_intr].fillna(0).astype(np.float32).values
tx_X_full = tx_features_df[feat_cols_full].fillna(0).astype(np.float32).values
agg_named = ["total_BTC","fees","num_input_addresses","num_output_addresses"]
agg_idxs_full = [feat_cols_full.index(c) for c in agg_named]
total_btc_idx_full = feat_cols_full.index("total_BTC")

addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) |
                      set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a:i for i,a in enumerate(wallet_addrs)}
N_w = len(wallet_addr_to_idx)
addr_tx_df["w"]  = addr_tx_df["input_address"].map(wallet_addr_to_idx)
addr_tx_df["tx"] = addr_tx_df["txId"].map(tx_id_to_idx)
addr_tx_df = addr_tx_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})
tx_addr_df["w"]  = tx_addr_df["output_address"].map(wallet_addr_to_idx)
tx_addr_df["tx"] = tx_addr_df["txId"].map(tx_id_to_idx)
tx_addr_df = tx_addr_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})

incident_in  = defaultdict(list)
incident_out = defaultdict(list)
for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values):
    incident_in[int(tx)].append(int(w))
for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values):
    incident_out[int(tx)].append(int(w))
wallet_in_txs  = defaultdict(list)
wallet_out_txs = defaultdict(list)
for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values):
    wallet_in_txs[int(w)].append((int(tx), int(tx_time[tx])))
for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values):
    wallet_out_txs[int(w)].append((int(tx), int(tx_time[tx])))
for w in wallet_in_txs:  wallet_in_txs[w].sort(key=lambda r:r[1])
for w in wallet_out_txs: wallet_out_txs[w].sort(key=lambda r:r[1])
all_incidence = defaultdict(set)
for w, lst in wallet_in_txs.items():
    for tx,_ in lst: all_incidence[w].add(tx)
for w, lst in wallet_out_txs.items():
    for tx,_ in lst: all_incidence[w].add(tx)

all_edges_master = pd.concat([
    addr_tx_df[["w","tx"]].assign(t=tx_time[addr_tx_df["tx"].values]),
    tx_addr_df[["w","tx"]].assign(t=tx_time[tx_addr_df["tx"].values]),
], ignore_index=True).drop_duplicates(subset=["w","tx"]).reset_index(drop=True)
print(f"  N_tx={N_tx:,}  N_w={N_w:,}  bipartite_edges={len(all_edges_master):,}")
print(f"  illicit={int((tx_label==1).sum()):,}")

Loading data...


  N_tx=203,769  N_w=822,942  bipartite_edges=1,268,260
  illicit=4,545


In [3]:
# Block C: Personalized PageRank seeded from illicit txs at τ < t.
#
# At each timestep t, build the cumulative bipartite adjacency over edges with
# tx_endpoint.t' <= t. Run PPR with a teleport vector concentrated uniformly on
# illicit txs with t' < t (strict — so we never seed on T's own timestep).
# Extract PPR scores for each tx at exactly t (and aggregate over its wallets).

print("Computing personalized PageRank features (Block C)...")
t0 = time.time()
N_total = N_w + N_tx

W_edges = all_edges_master["w"].values.astype(np.int64)
T_edges = all_edges_master["tx"].values.astype(np.int64)
T_times = all_edges_master["t"].values.astype(np.int64)
edge_order = np.argsort(T_times)
W_sorted = W_edges[edge_order]
T_sorted = T_edges[edge_order]
Tt_sorted = T_times[edge_order]

# Output arrays
ppr_self      = np.zeros(N_tx, dtype=np.float32)
ppr_max_nbr   = np.zeros(N_tx, dtype=np.float32)
ppr_mean_nbr  = np.zeros(N_tx, dtype=np.float32)
ppr_n_seeds   = np.zeros(N_tx, dtype=np.float32)

incident_in_arr  = {tx: list(set(ws)) for tx, ws in incident_in.items()}
incident_out_arr = {tx: list(set(ws)) for tx, ws in incident_out.items()}

for t in range(1, N_TIME_STEPS + 1):
    cut = int(np.searchsorted(Tt_sorted, t, side="right"))
    if cut == 0:
        continue
    we = W_sorted[:cut]
    te = T_sorted[:cut]
    rows = np.concatenate([we, N_w + te])
    cols = np.concatenate([N_w + te, we])
    data = np.ones(rows.shape[0], dtype=np.float32)
    A = sp.csr_matrix((data, (rows, cols)), shape=(N_total, N_total))
    deg = np.asarray(A.sum(axis=1)).flatten()
    inv_deg = np.zeros_like(deg)
    nz = deg > 0; inv_deg[nz] = 1.0 / deg[nz]

    # Build seed: illicit txs with t' < t (strict). At t=1 the seed is empty.
    illicit_seed_idx = np.where((tx_label == 1) & (tx_time < t))[0]
    if illicit_seed_idx.size == 0:
        # No seeds yet — leave zeros.
        # (still need to record n_seeds for txs at t)
        for tx_idx in np.where(tx_time == t)[0]:
            ppr_n_seeds[tx_idx] = 0.0
        continue
    s = np.zeros(N_total, dtype=np.float32)
    s[N_w + illicit_seed_idx] = 1.0 / illicit_seed_idx.size

    teleport = (1.0 - PPR_DAMP) * s
    pr = s.copy()
    for _ in range(PPR_ITERATIONS):
        weighted = pr * inv_deg
        pr = teleport + PPR_DAMP * (A @ weighted).astype(np.float32)

    txs_at_t = np.where(tx_time == t)[0]
    for tx_idx in txs_at_t:
        in_w  = incident_in_arr.get(int(tx_idx),  [])
        out_w = incident_out_arr.get(int(tx_idx), [])
        all_nbr = list(set(in_w) | set(out_w))
        ppr_self[tx_idx]    = float(pr[N_w + tx_idx])
        ppr_n_seeds[tx_idx] = float(illicit_seed_idx.size)
        if all_nbr:
            v = pr[all_nbr]
            ppr_max_nbr[tx_idx]  = float(v.max())
            ppr_mean_nbr[tx_idx] = float(v.mean())
    if t % 5 == 0 or t == N_TIME_STEPS:
        print(f"  t={t:>2d}  edges={cut:>9,}  seeds={int(illicit_seed_idx.size):>4,}  ({time.time()-t0:.1f}s)")

ppr_self_log = np.log1p(ppr_self * 1e6).astype(np.float32)  # rescale for stability
block_C_X = np.stack([ppr_self, ppr_self_log, ppr_max_nbr, ppr_mean_nbr, ppr_n_seeds], axis=1).astype(np.float32)
BLOCK_C_FEATS = ["ppr_illicit","ppr_illicit_log","max_nbr_ppr_illicit","mean_nbr_ppr_illicit","n_illicit_seeds_at_t"]
F_C = block_C_X.shape[1]
print(f"  Block C done in {time.time()-t0:.1f}s.  block_C_X={block_C_X.shape}")
print(f"  ppr_self stats: min={ppr_self.min():.2e}  median={np.median(ppr_self):.2e}  max={ppr_self.max():.2e}")

Computing personalized PageRank features (Block C)...


  t= 5  edges=  189,571  seeds=  76  (1.1s)


  t=10  edges=  349,941  seeds= 506  (2.0s)


  t=15  edges=  450,202  seeds=1,005  (3.1s)


  t=20  edges=  546,600  seeds=1,511  (4.2s)


...

  t=30  edges=  767,134  seeds=2,871  (7.2s)


  t=35  edges=  873,958  seeds=3,462  (8.6s)


  t=40  edges=1,012,737  seeds=3,909  (10.2s)


  t=45  edges=1,179,046  seeds=4,424  (11.8s)


  t=49  edges=1,268,260  seeds=4,489  (13.4s)
  Block C done in 13.4s.  block_C_X=(203769, 5)
  ppr_self stats: min=0.00e+00  median=4.29e-09  max=6.66e-04


In [4]:
# Need the cross-step tx->tx graph for Block D motifs.
print(f"Building cross-step tx→tx edges (cap={MAX_WALLET_DEGREE})...")
ROLE_OUT_IN, ROLE_OUT_OUT, ROLE_IN_IN, ROLE_IN_OUT = 0,1,2,3
src_l, dst_l, dt_l, role_l = [], [], [], []
for w in all_incidence:
    if len(all_incidence[w]) < 2 or len(all_incidence[w]) > MAX_WALLET_DEGREE: continue
    events = []
    for tx, t in wallet_in_txs.get(w, []):  events.append((t, tx, 'in'))
    for tx, t in wallet_out_txs.get(w, []): events.append((t, tx, 'out'))
    events.sort(key=lambda r: r[0])
    K = len(events)
    for i in range(K):
        ti, txi, si = events[i]
        for j in range(i+1, K):
            tj, txj, sj = events[j]
            if tj <= ti or txi == txj: continue
            if   si=='out' and sj=='in':  role = ROLE_OUT_IN
            elif si=='out' and sj=='out': role = ROLE_OUT_OUT
            elif si=='in'  and sj=='in':  role = ROLE_IN_IN
            else:                          role = ROLE_IN_OUT
            src_l.append(txi); dst_l.append(txj)
            dt_l.append(tj-ti); role_l.append(role)
src = np.array(src_l, dtype=np.int64); dst = np.array(dst_l, dtype=np.int64)
dt_edges = np.array(dt_l, dtype=np.int32); role_e = np.array(role_l, dtype=np.int8)
E = len(src); print(f"  E={E:,}")

# Build incoming-edge index (for fast 'edges into tx j' lookup)
order_in = np.argsort(dst, kind="stable")
dst_in = dst[order_in]; src_in = src[order_in]
dt_in  = dt_edges[order_in]; role_in = role_e[order_in]
in_offsets = np.searchsorted(dst_in, np.arange(N_tx + 1)).astype(np.int64)

# Build outgoing-edge index (for 'edges out of tx i' lookup)
order_out = np.argsort(src, kind="stable")
src_out = src[order_out]; dst_out = dst[order_out]
dt_out  = dt_edges[order_out]; role_out = role_e[order_out]
out_offsets = np.searchsorted(src_out, np.arange(N_tx + 1)).astype(np.int64)
print(f"  in/out indices built")

Building cross-step tx→tx edges (cap=50)...


  E=445,180
  in/out indices built


In [5]:
# Block D: 2-hop temporal motifs T'' -> T' -> T (chained cross-step edges).
# For each T at time t we look at incoming edges T' -> T,
# then for each such T' we look at incoming edges T'' -> T' (with t(T'') < t(T')).
# Counts:
#   D0  total 2-hop count
#   D1  count where T'' is illicit
#   D2  count where T' is illicit
#   D3  count where both edges are out_in (clean money-flow chain)
#   D4  count where total dt = (t - t(T'')) <= 5  (recent)
#   D5  decayed: sum exp(-beta * (t - t(T''))) * 1[T'' illicit]

print("Computing 2-hop temporal motifs (Block D)...")
t0 = time.time()
BLOCK_D_FEATS = ["motif2_total","motif2_T2pp_illicit","motif2_T1p_illicit",
                 "motif2_both_outin","motif2_recent_le5","motif2_decayed_T2pp_illicit"]
F_D = len(BLOCK_D_FEATS)
block_D_X = np.zeros((N_tx, F_D), dtype=np.float32)

for j in range(N_tx):
    a, b = in_offsets[j], in_offsets[j+1]
    if a == b: continue
    t_T = int(tx_time[j])
    primes = src_in[a:b]
    role_first = role_in[a:b]
    total = 0; cnt_t2pp_ill = 0; cnt_t1p_ill = 0
    cnt_both_outin = 0; cnt_recent = 0; dec_t2pp_ill = 0.0
    for k in range(b - a):
        T1 = int(primes[k])
        r1 = int(role_first[k])
        a2, b2 = in_offsets[T1], in_offsets[T1+1]
        if a2 == b2: continue
        T2pp = src_in[a2:b2]
        r2   = role_in[a2:b2]
        # we'll also use the absolute time for decay
        t_T2pp = tx_time[T2pp]
        # safety: only count distinct T''  (not equal to T)
        valid = (T2pp != j) & (t_T2pp < int(tx_time[T1]))
        if not valid.any(): continue
        T2pp_v = T2pp[valid]; r2_v = r2[valid]; tT2_v = t_T2pp[valid]
        n_v = T2pp_v.size
        total += n_v
        ill_T2pp_mask = (tx_label[T2pp_v] == 1)
        cnt_t2pp_ill += int(ill_T2pp_mask.sum())
        cnt_t1p_ill  += n_v if tx_label[T1] == 1 else 0
        cnt_both_outin += int(((r2_v == ROLE_OUT_IN) & (r1 == ROLE_OUT_IN)).sum())
        cnt_recent += int(((t_T - tT2_v) <= 5).sum())
        if ill_T2pp_mask.any():
            dt2 = (t_T - tT2_v[ill_T2pp_mask]).astype(np.float64)
            dec_t2pp_ill += float(np.exp(-MOTIF_DECAY * dt2).sum())
    block_D_X[j, 0] = total
    block_D_X[j, 1] = cnt_t2pp_ill
    block_D_X[j, 2] = cnt_t1p_ill
    block_D_X[j, 3] = cnt_both_outin
    block_D_X[j, 4] = cnt_recent
    block_D_X[j, 5] = dec_t2pp_ill
    if j > 0 and j % 50_000 == 0:
        print(f"  tx {j:,}/{N_tx:,}  ({time.time()-t0:.0f}s)")

print(f"  Block D done in {time.time()-t0:.0f}s.  block_D_X={block_D_X.shape}")
print("  motif counts (full set):")
for i, nm in enumerate(BLOCK_D_FEATS):
    v = block_D_X[:, i]
    nz = int((v > 0).sum())
    print(f"    {nm:35s}  nonzero={nz:>6,}  max={v.max():.2f}  mean(nz)={v[v>0].mean() if nz else 0:.2f}")

Computing 2-hop temporal motifs (Block D)...


  Block D done in 5s.  block_D_X=(203769, 6)
  motif counts (full set):
    motif2_total                         nonzero=22,819  max=68497296.00  mean(nz)=11704.71
    motif2_T2pp_illicit                  nonzero= 1,150  max=2868.00  mean(nz)=6.63
    motif2_T1p_illicit                   nonzero=   196  max=90.00  mean(nz)=10.95
    motif2_both_outin                    nonzero= 4,460  max=1077.00  mean(nz)=39.66
    motif2_recent_le5                    nonzero=13,874  max=763714.00  mean(nz)=525.55
    motif2_decayed_T2pp_illicit          nonzero= 1,150  max=42.98  mean(nz)=0.61


In [6]:
# Rebuild Phase 1 traj features (103) and Block B pair feats (17) for the ablation.
# This duplicates step2 logic so this notebook is self-contained.
MAX_INCIDENT_PER_SIDE = 32
MAX_CO_WALLETS        = 150
RECENCY_SENTINEL      = N_TIME_STEPS * 2
TRAJ_DECAY            = 0.2
DECAY_BETA            = 0.2

all_edges_w = pd.concat([
    addr_tx_df[["w","tx"]].assign(t=tx_time[addr_tx_df["tx"].values],
                                  tx_lab=tx_label[addr_tx_df["tx"].values]),
    tx_addr_df[["w","tx"]].assign(t=tx_time[tx_addr_df["tx"].values],
                                  tx_lab=tx_label[tx_addr_df["tx"].values]),
], ignore_index=True).drop_duplicates(subset=["w","tx"]).sort_values(["w","t"]).reset_index(drop=True)
g = all_edges_w.groupby("w", sort=False)
wallet_t_arr = {}; wallet_tx_arr = {}; wallet_lab_arr = {}
for w, sub in g:
    wallet_t_arr[int(w)]   = sub["t"].values.astype(np.int64)
    wallet_tx_arr[int(w)]  = sub["tx"].values.astype(np.int64)
    wallet_lab_arr[int(w)] = sub["tx_lab"].values.astype(np.int64)
wallet_event_count = {w:len(t) for w, t in wallet_t_arr.items()}
wallet_has_illicit_by = {}
for w, labs in wallet_lab_arr.items():
    m = (labs == 1)
    if m.any():
        wallet_has_illicit_by[w] = int(wallet_t_arr[w][m].min())

def pick_top_wallets(wlist, k=MAX_INCIDENT_PER_SIDE):
    if len(wlist) <= k: return list(wlist)
    cnts = np.array([wallet_event_count.get(w,0) for w in wlist], dtype=np.int64)
    order = np.argsort(-cnts, kind="stable")
    return [wlist[i] for i in order[:k]]

def per_wallet_priors(w, t_query):
    tarr = wallet_t_arr.get(w)
    if tarr is None: return None
    cut = int(np.searchsorted(tarr, t_query, side="left"))
    if cut == 0: return None
    prior_t = tarr[:cut]; prior_lab = wallet_lab_arr[w][:cut]; prior_tx = wallet_tx_arr[w][:cut]
    illicit_mask = (prior_lab==1)
    n_illicit = int(illicit_mask.sum()); n_licit = int((prior_lab==0).sum())
    last_illicit_t = int(prior_t[illicit_mask].max()) if n_illicit>0 else -1
    if n_illicit > 0:
        decay_w = np.exp(-TRAJ_DECAY*(t_query-prior_t[illicit_mask]).astype(np.float64))
        decayed_illicit_score = float(decay_w.sum())
    else:
        decayed_illicit_score = 0.0
    illicit_frac = n_illicit/max(n_illicit+n_licit,1)
    co_wallets = set()
    for tx_i in prior_tx:
        for cw in incident_in.get(int(tx_i),[]):
            if cw != w: co_wallets.add(cw)
        for cw in incident_out.get(int(tx_i),[]):
            if cw != w: co_wallets.add(cw)
        if len(co_wallets) >= MAX_CO_WALLETS: break
    n_co_wallets = len(co_wallets)
    n_co_illicit = sum(1 for cw in co_wallets if wallet_has_illicit_by.get(cw, N_TIME_STEPS+1) < t_query)
    uin, uout = set(), set()
    for tx_i in prior_tx:
        for cw in incident_in.get(int(tx_i),[]):
            if cw != w: uin.add(cw)
        for cw in incident_out.get(int(tx_i),[]):
            if cw != w: uout.add(cw)
    n_inp = len(uin); n_outp = len(uout)
    fan_asym = n_outp/max(n_inp,1)
    age = int(t_query - prior_t[0]); recency = int(t_query - prior_t[-1])
    n_recent_5 = int(((t_query-prior_t)<=5).sum()); n_recent_1 = int(((t_query-prior_t)<=1).sum())
    if cut>=2:
        iat = np.diff(prior_t.astype(np.float64))
        iat_mean = float(iat.mean()); iat_std = float(iat.std())
        burstiness = float((iat_std-iat_mean)/(iat_std+iat_mean+1e-8))
    else: iat_mean=0.0; iat_std=0.0; burstiness=0.0
    velocity = n_recent_5/max(cut,1)
    feat_vals = tx_X_full[prior_tx][:, agg_idxs_full]
    return {"n":int(cut),"n_illicit":n_illicit,"n_licit":n_licit,"illicit_frac":illicit_frac,
            "last_illicit_t":last_illicit_t,"decayed_illicit":decayed_illicit_score,
            "n_co_wallets":n_co_wallets,"n_co_illicit":n_co_illicit,
            "co_illicit_frac":n_co_illicit/max(n_co_wallets,1),
            "n_in_partners":n_inp,"n_out_partners":n_outp,"fan_asymmetry":fan_asym,
            "first_seen_t":int(prior_t[0]),"last_seen_t":int(prior_t[-1]),
            "age":age,"recency":recency,"n_recent_5":n_recent_5,"n_recent_1":n_recent_1,
            "iat_mean":iat_mean,"iat_std":iat_std,"burstiness":burstiness,"velocity":velocity,
            "feat_means":feat_vals.mean(axis=0),"feat_maxes":feat_vals.max(axis=0)}

def aggregate_side(summaries, p, t_T):
    n_total = len(summaries); valid = [s for s in summaries if s is not None]; n_w_prior = len(valid)
    out = {f"{p}_n_wallets":n_total, f"{p}_n_wallets_with_prior":n_w_prior,
           f"{p}_frac_first_appearance":(n_total-n_w_prior)/max(n_total,1)}
    keys_zero = ["n_priors_sum","n_priors_max","n_illicit_sum","n_illicit_max","n_licit_sum",
                 "n_wallets_with_illicit","n_wallets_illicit_frac_gt0","n_wallets_illicit_frac_gt50",
                 "illicit_frac_max","illicit_frac_mean","decayed_illicit_max","decayed_illicit_sum",
                 "recency_to_illicit_min","co_illicit_sum","co_illicit_max","co_illicit_frac_max",
                 "co_illicit_frac_mean","n_co_wallets_sum","fan_asymmetry_max","fan_asymmetry_mean",
                 "n_in_partners_max","n_out_partners_max","frac_single_use","age_max","age_mean",
                 "recency_min","n_recent_5_sum","n_recent_5_max","n_recent_1_sum","velocity_max",
                 "velocity_mean","burstiness_max","burstiness_mean","iat_mean_min","iat_std_max"]
    for k in keys_zero: out[f"{p}_{k}"] = 0.0
    out[f"{p}_recency_to_illicit_min"] = float(RECENCY_SENTINEL)
    out[f"{p}_recency_min"] = float(RECENCY_SENTINEL)
    for nm in agg_named:
        out[f"{p}_prior_{nm}_mean_max"] = 0.0
        out[f"{p}_prior_{nm}_max_max"]  = 0.0
    if not valid: return out
    ns=np.array([s["n"] for s in valid],dtype=np.float64)
    nis=np.array([s["n_illicit"] for s in valid],dtype=np.float64)
    nls=np.array([s["n_licit"] for s in valid],dtype=np.float64)
    ill_frac=np.array([s["illicit_frac"] for s in valid],dtype=np.float64)
    dec_ill=np.array([s["decayed_illicit"] for s in valid],dtype=np.float64)
    last_ill=np.array([s["last_illicit_t"] for s in valid],dtype=np.int64)
    has_ill=(last_ill>=0)
    rec_ill=np.where(has_ill, t_T-last_ill, RECENCY_SENTINEL).astype(np.float64)
    co_ill=np.array([s["n_co_illicit"] for s in valid],dtype=np.float64)
    co_n =np.array([s["n_co_wallets"] for s in valid],dtype=np.float64)
    co_fr=np.array([s["co_illicit_frac"] for s in valid],dtype=np.float64)
    fan_a=np.array([s["fan_asymmetry"] for s in valid],dtype=np.float64)
    n_inp=np.array([s["n_in_partners"] for s in valid],dtype=np.float64)
    n_outp=np.array([s["n_out_partners"] for s in valid],dtype=np.float64)
    ages=np.array([s["age"] for s in valid],dtype=np.float64)
    recs=np.array([s["recency"] for s in valid],dtype=np.float64)
    nr5=np.array([s["n_recent_5"] for s in valid],dtype=np.float64)
    nr1=np.array([s["n_recent_1"] for s in valid],dtype=np.float64)
    vel=np.array([s["velocity"] for s in valid],dtype=np.float64)
    bur=np.array([s["burstiness"] for s in valid],dtype=np.float64)
    iam=np.array([s["iat_mean"] for s in valid],dtype=np.float64)
    ias=np.array([s["iat_std"] for s in valid],dtype=np.float64)
    fm=np.stack([s["feat_means"] for s in valid],axis=0)
    fx=np.stack([s["feat_maxes"] for s in valid],axis=0)
    out[f"{p}_n_priors_sum"]=float(ns.sum()); out[f"{p}_n_priors_max"]=float(ns.max())
    out[f"{p}_n_illicit_sum"]=float(nis.sum()); out[f"{p}_n_illicit_max"]=float(nis.max())
    out[f"{p}_n_licit_sum"]=float(nls.sum())
    out[f"{p}_n_wallets_with_illicit"]=float(has_ill.sum())
    out[f"{p}_n_wallets_illicit_frac_gt0"]=float((ill_frac>0).sum())
    out[f"{p}_n_wallets_illicit_frac_gt50"]=float((ill_frac>0.5).sum())
    out[f"{p}_illicit_frac_max"]=float(ill_frac.max()); out[f"{p}_illicit_frac_mean"]=float(ill_frac.mean())
    out[f"{p}_decayed_illicit_max"]=float(dec_ill.max()); out[f"{p}_decayed_illicit_sum"]=float(dec_ill.sum())
    out[f"{p}_recency_to_illicit_min"]=float(rec_ill.min())
    out[f"{p}_co_illicit_sum"]=float(co_ill.sum()); out[f"{p}_co_illicit_max"]=float(co_ill.max())
    out[f"{p}_co_illicit_frac_max"]=float(co_fr.max()); out[f"{p}_co_illicit_frac_mean"]=float(co_fr.mean())
    out[f"{p}_n_co_wallets_sum"]=float(co_n.sum())
    out[f"{p}_fan_asymmetry_max"]=float(fan_a.max()); out[f"{p}_fan_asymmetry_mean"]=float(fan_a.mean())
    out[f"{p}_n_in_partners_max"]=float(n_inp.max()); out[f"{p}_n_out_partners_max"]=float(n_outp.max())
    out[f"{p}_frac_single_use"]=sum(1 for s in valid if s["n"]==1)/max(n_w_prior,1)
    out[f"{p}_age_max"]=float(ages.max()); out[f"{p}_age_mean"]=float(ages.mean())
    out[f"{p}_recency_min"]=float(recs.min())
    out[f"{p}_n_recent_5_sum"]=float(nr5.sum()); out[f"{p}_n_recent_5_max"]=float(nr5.max())
    out[f"{p}_n_recent_1_sum"]=float(nr1.sum())
    out[f"{p}_velocity_max"]=float(vel.max()); out[f"{p}_velocity_mean"]=float(vel.mean())
    out[f"{p}_burstiness_max"]=float(bur.max()); out[f"{p}_burstiness_mean"]=float(bur.mean())
    out[f"{p}_iat_mean_min"]=float(iam.min()); out[f"{p}_iat_std_max"]=float(ias.max())
    for k, nm in enumerate(agg_named):
        out[f"{p}_prior_{nm}_mean_max"]=float(fm[:,k].max())
        out[f"{p}_prior_{nm}_max_max"]=float(fx[:,k].max())
    return out

print("Computing 103 traj features...")
t0 = time.time(); traj_rows = []
for tx_idx in range(N_tx):
    t_T = int(tx_time[tx_idx])
    in_w  = pick_top_wallets(incident_in.get(tx_idx, []))
    out_w = pick_top_wallets(incident_out.get(tx_idx, []))
    in_summ  = [per_wallet_priors(w, t_T) for w in in_w]
    out_summ = [per_wallet_priors(w, t_T) for w in out_w]
    row = {}; row.update(aggregate_side(in_summ,"in",t_T)); row.update(aggregate_side(out_summ,"out",t_T))
    row["both_sides_have_illicit"] = float(row["in_n_wallets_with_illicit"]>0 and row["out_n_wallets_with_illicit"]>0)
    row["total_n_illicit_priors"]  = row["in_n_illicit_sum"]+row["out_n_illicit_sum"]
    row["total_n_wallets_with_illicit"] = row["in_n_wallets_with_illicit"]+row["out_n_wallets_with_illicit"]
    row["total_co_illicit"] = row["in_co_illicit_sum"]+row["out_co_illicit_sum"]
    row["min_recency_to_illicit"] = min(row["in_recency_to_illicit_min"], row["out_recency_to_illicit_min"])
    row["max_illicit_frac_either_side"] = max(row["in_illicit_frac_max"], row["out_illicit_frac_max"])
    row["max_decayed_illicit_either"]   = max(row["in_decayed_illicit_max"], row["out_decayed_illicit_max"])
    row["max_co_illicit_frac_either"]   = max(row["in_co_illicit_frac_max"], row["out_co_illicit_frac_max"])
    row["total_frac_first_appearance"] = (
        (row["in_frac_first_appearance"]*max(row["in_n_wallets"],1) +
         row["out_frac_first_appearance"]*max(row["out_n_wallets"],1))/
        max(row["in_n_wallets"]+row["out_n_wallets"],1))
    T_btc = float(tx_X_full[tx_idx, total_btc_idx_full])
    max_p = max(row["in_prior_total_BTC_max_max"], row["out_prior_total_BTC_max_max"])
    mean_p = max(row["in_prior_total_BTC_mean_max"], row["out_prior_total_BTC_mean_max"])
    row["T_btc_vs_max_prior"]  = T_btc/max(max_p,1.0)
    row["T_btc_vs_mean_prior"] = T_btc/max(mean_p,1.0)
    traj_rows.append(row)
traj_df = pd.DataFrame(traj_rows); traj_X = traj_df.values.astype(np.float32)
F_TRAJ = traj_X.shape[1]
print(f"  done in {time.time()-t0:.0f}s.  traj_X={traj_X.shape}")

# Block B (pair feats — money-flow only): same as step2 A3.
print("\nComputing Block B (money-flow only pair features)...")
t0 = time.time()
BLOCK_B_FEATS = ["B_n_in_edges","B_n_unique_src","B_dt_min","B_dt_mean","B_dt_max","B_decayed_mass",
                 "B_n_illicit_src","B_n_licit_src","B_frac_illicit_src","B_decayed_illicit",
                 "B_src_total_BTC_mean","B_src_total_BTC_max",
                 "B_src_fees_mean","B_src_fees_max",
                 "B_src_n_in_addr_mean","B_src_n_out_addr_mean","B_max_src_multiplicity"]
F_B = len(BLOCK_B_FEATS)
block_B_X = np.zeros((N_tx, F_B), dtype=np.float32)
src_total_btc_arr = tx_X_full[:, total_btc_idx_full]
src_agg_feats_arr = tx_X_full[:, agg_idxs_full]
for j in range(N_tx):
    a, b = in_offsets[j], in_offsets[j+1]
    if a == b: continue
    s_arr=src_in[a:b]; dt_a=dt_in[a:b]; r_a=role_in[a:b]
    bm = (r_a == ROLE_OUT_IN)
    if not bm.any(): continue
    s_B=s_arr[bm]; dt_B=dt_a[bm]; lab_B=tx_label[s_B]
    dec_B = np.exp(-DECAY_BETA*dt_B.astype(np.float64))
    af_B  = src_agg_feats_arr[s_B]
    uniq_B, mult_B = np.unique(s_B, return_counts=True)
    ill = (lab_B==1); lic = (lab_B==0)
    n_ill=int(ill.sum()); n_lic=int(lic.sum()); n_lab=n_ill+n_lic
    block_B_X[j, 0]  = bm.sum()
    block_B_X[j, 1]  = uniq_B.size
    block_B_X[j, 2]  = float(dt_B.min())
    block_B_X[j, 3]  = float(dt_B.mean())
    block_B_X[j, 4]  = float(dt_B.max())
    block_B_X[j, 5]  = float(dec_B.sum())
    block_B_X[j, 6]  = n_ill
    block_B_X[j, 7]  = n_lic
    block_B_X[j, 8]  = (n_ill/n_lab) if n_lab>0 else 0.0
    block_B_X[j, 9]  = float(dec_B[ill].sum()) if n_ill>0 else 0.0
    block_B_X[j,10]  = float(af_B[:,0].mean()); block_B_X[j,11] = float(af_B[:,0].max())
    block_B_X[j,12]  = float(af_B[:,1].mean()); block_B_X[j,13] = float(af_B[:,1].max())
    block_B_X[j,14]  = float(af_B[:,2].mean()); block_B_X[j,15] = float(af_B[:,3].mean())
    block_B_X[j,16]  = int(mult_B.max())
print(f"  done in {time.time()-t0:.1f}s.")

Computing 103 traj features...


  done in 42s.  traj_X=(203769, 103)

Computing Block B (money-flow only pair features)...


  done in 0.5s.


In [7]:
labelled = (tx_label != -1)
train_mask = labelled & (tx_time <= TRAIN_END)
test_mask  = labelled & (tx_time >  TRAIN_END)
y_train = tx_label[train_mask]; y_test = tx_label[test_mask]
test_t_arr = tx_time[test_mask]
print(f"train: n={int(train_mask.sum()):,}  illicit_rate={y_train.mean():.4f}")
print(f"test:  n={int(test_mask.sum()):,}  illicit_rate={y_test.mean():.4f}")

X_intr_tr = tx_X_intr[train_mask]; X_intr_te = tx_X_intr[test_mask]
traj_tr = traj_X[train_mask]; traj_te = traj_X[test_mask]
B_tr = block_B_X[train_mask];   B_te = block_B_X[test_mask]
C_tr = block_C_X[train_mask];   C_te = block_C_X[test_mask]
D_tr = block_D_X[train_mask];   D_te = block_D_X[test_mask]

ablations = {
    f"A0 [108]":            (X_intr_tr, X_intr_te),
    f"A1 [+103 traj]":      (np.concatenate([X_intr_tr, traj_tr],axis=1),
                              np.concatenate([X_intr_te, traj_te],axis=1)),
    f"A3 [+B (17)]":        (np.concatenate([X_intr_tr, traj_tr, B_tr],axis=1),
                              np.concatenate([X_intr_te, traj_te, B_te],axis=1)),
    f"C1 [A3 +C ({F_C})]":  (np.concatenate([X_intr_tr, traj_tr, B_tr, C_tr],axis=1),
                              np.concatenate([X_intr_te, traj_te, B_te, C_te],axis=1)),
    f"C2 [A3 +D ({F_D})]":  (np.concatenate([X_intr_tr, traj_tr, B_tr, D_tr],axis=1),
                              np.concatenate([X_intr_te, traj_te, B_te, D_te],axis=1)),
    f"C3 [A3 +C+D ({F_C+F_D})]": (np.concatenate([X_intr_tr, traj_tr, B_tr, C_tr, D_tr],axis=1),
                                   np.concatenate([X_intr_te, traj_te, B_te, C_te, D_te],axis=1)),
}

results = {}; preds = {}; last_rf = None
for name, (Xtr, Xte) in ablations.items():
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 n_jobs=-1, random_state=RANDOM_SEED)
    rf.fit(Xtr, y_train)
    yp = rf.predict(Xte); ypr = rf.predict_proba(Xte)[:,1]
    f1=f1_score(y_test,yp,pos_label=1,zero_division=0)
    auc=roc_auc_score(y_test,ypr); pr=average_precision_score(y_test,ypr)
    results[name]=(f1,auc,pr,Xtr.shape[1]); preds[name]=yp
    print(f"  [{name}]  dim={Xtr.shape[1]}  F1={f1:.4f}  AUC={auc:.4f}  PR-AUC={pr:.4f}  ({time.time()-t0:.1f}s)")
    last_rf = rf

train: n=29,894  illicit_rate=0.1158
test:  n=16,670  illicit_rate=0.0650


  [A0 [108]]  dim=108  F1=0.8021  AUC=0.9026  PR-AUC=0.7855  (3.0s)


  [A1 [+103 traj]]  dim=211  F1=0.8098  AUC=0.9196  PR-AUC=0.8029  (2.8s)


  [A3 [+B (17)]]  dim=228  F1=0.8122  AUC=0.9160  PR-AUC=0.8027  (2.7s)


  [C1 [A3 +C (5)]]  dim=233  F1=0.8149  AUC=0.9209  PR-AUC=0.8067  (2.7s)


  [C2 [A3 +D (6)]]  dim=234  F1=0.8099  AUC=0.9200  PR-AUC=0.8035  (2.6s)


  [C3 [A3 +C+D (11)]]  dim=239  F1=0.8114  AUC=0.9233  PR-AUC=0.8075  (3.2s)


In [8]:
print("="*75)
print(f"{'Model':35s}  {'dim':>5s}  {'F1':>7s}  {'AUC':>7s}  {'PR-AUC':>7s}")
print("="*75)
for n,(f1,auc,pr,d) in results.items():
    print(f"  {n:33s}  {d:>5d}  {f1:>7.4f}  {auc:>7.4f}  {pr:>7.4f}")

f1_a3 = next(v[0] for k,v in results.items() if k.startswith("A3"))
print()
for k,v in results.items():
    if k.startswith("C"):
        print(f"  Δ F1 vs A3: {k}  →  {v[0]-f1_a3:+.4f}")

# Per-timestep
print("\n"+"="*75)
print("Per-timestep test F1 (illicit class)")
print("="*75)
names = list(ablations.keys())
header = f"{'t':>3}  {'n':>5}  {'ill':>4}" + "".join(f"  {n[:14]:>14s}" for n in names)
print(header)
for t in range(TRAIN_END+1, N_TIME_STEPS+1):
    m = (test_t_arr==t)
    if not m.any(): continue
    yt = y_test[m]
    cells = []
    for n in names:
        f1t = f1_score(yt, preds[n][m], pos_label=1, zero_division=0) if yt.sum()>0 else float("nan")
        cells.append(f"  {f1t:>14.4f}")
    print(f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>4}" + "".join(cells))

# Top-30 importance from C3
print("\n"+"="*75)
print("Top-30 features in C3")
print("="*75)
all_names = (list(feat_cols_intr) + list(traj_df.columns) + BLOCK_B_FEATS + BLOCK_C_FEATS + BLOCK_D_FEATS)
imp = last_rf.feature_importances_
order = np.argsort(-imp)[:30]
F_off_traj = F_INTRINSIC + F_TRAJ
F_off_B = F_off_traj + F_B
F_off_C = F_off_B + F_C
n_traj=n_b=n_c=n_d=0
for rank, i in enumerate(order, 1):
    if i < F_INTRINSIC: tag = ""
    elif i < F_off_traj: tag = "  (TRAJ)"; n_traj += 1
    elif i < F_off_B:    tag = "  (B)";    n_b += 1
    elif i < F_off_C:    tag = "  (C-PPR)"; n_c += 1
    else:                tag = "  (D-MOTIF)"; n_d += 1
    print(f"  {rank:2d}.  {imp[i]:.4f}  {all_names[i]}{tag}")
print(f"\n  in top-30: {n_traj} TRAJ / {n_b} B / {n_c} C-PPR / {n_d} D-MOTIF")

Model                                  dim       F1      AUC   PR-AUC
  A0 [108]                             108   0.8021   0.9026   0.7855
  A1 [+103 traj]                       211   0.8098   0.9196   0.8029
  A3 [+B (17)]                         228   0.8122   0.9160   0.8027
  C1 [A3 +C (5)]                       233   0.8149   0.9209   0.8067
  C2 [A3 +D (6)]                       234   0.8099   0.9200   0.8035
  C3 [A3 +C+D (11)]                    239   0.8114   0.9233   0.8075

  Δ F1 vs A3: C1 [A3 +C (5)]  →  +0.0027
  Δ F1 vs A3: C2 [A3 +D (6)]  →  -0.0023
  Δ F1 vs A3: C3 [A3 +C+D (11)]  →  -0.0008

Per-timestep test F1 (illicit class)
  t      n   ill        A0 [108]  A1 [+103 traj]    A3 [+B (17)]  C1 [A3 +C (5)]  C2 [A3 +D (6)]  C3 [A3 +C+D (1
 35   1341   182          0.9607          0.9636          0.9609          0.9663          0.9609          0.9663
 36   1708    33          0.9143          0.9143          0.9296          0.9429          0.9296          0.9552
 37   